# The "not found" gate

Nov 1. When should Mixray answer that a sound is not in the clip?

**Data:** AlbumDB. Every case produced all six Demucs stems, and each mixture's make-up is known, so stems of instruments never mixed in are real absent examples.

**What it decided:** a stem under -25 dB is absent: 31 of 31 absent stems caught, 8 of 89 present ones flagged.

This is a selection of the cells from that work, with the output they printed. They rely on earlier cells and on a Drive folder of datasets and outputs that is not in this repository.


## The gate


In [ ]:
"""D6: can the product tell, at runtime, that it failed to find the requested sound?

The only signals allowed here are ones a request actually has: the served stem's own level against
the mixture, and how many notes came out. No clean reference, no oracle.
"""

GATE = r'''
"""Measure runtime-visible signals per case and pair them with the quality already measured."""

import json
import sys
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

RATE = 48000
PROMPT_STEM_KEYWORDS = (
    ("vocals", {"vocal", "vocals", "voice", "sing", "singing", "singer"}),
    ("drums", {"drum", "drums", "kick", "snare", "hihat", "hat", "cymbal"}),
    ("bass", {"bass", "sub", "subbass"}),
    ("guitar", {"guitar", "guitars"}),
    ("piano", {"piano", "keys", "keyboard"}),
)


def stem_for_prompt(prompt):
    tokens = set(prompt.lower().split())
    for stem, keywords in PROMPT_STEM_KEYWORDS:
        if tokens & keywords:
            return stem
    return None


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def main():
    rows = []
    for run_dir in sys.argv[1:]:
        run = Path(run_dir)
        for case in json.loads((run / "config.json").read_text())["cases"]:
            case_dir = run / case["id"]
            if not (case_dir / "mix.wav").is_file():
                continue
            requested = stem_for_prompt(case["prompt"])
            mixture = mono(case_dir / "mix.wav")
            served = None
            if requested:
                path = case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{requested}.wav"
                if path.is_file():
                    served = mono(path, len(mixture))
            if served is None:
                continue
            mix_rms = float(np.sqrt(np.mean(mixture**2)))
            stem_rms = float(np.sqrt(np.mean(served**2)))
            # Energy kept by the served stem, and how peaky it is: an empty stem is quiet and flat.
            rows.append({
                "case": case["id"], "prompt": case["prompt"], "stem": requested,
                "stem_to_mix_db": 20.0 * np.log10(max(stem_rms, 1e-12) / max(mix_rms, 1e-12)),
                "stem_peak": float(np.max(np.abs(served))),
                "stem_crest_db": 20.0 * np.log10(
                    max(float(np.max(np.abs(served))), 1e-12) / max(stem_rms, 1e-12)
                ),
            })
    Path("/content/gate_signals.json").write_text(json.dumps(rows, indent=2))
    print("GATE_DONE " + json.dumps({"cases": len(rows)}))


if __name__ == "__main__":
    main()
'''

Path("/content/gate_signals.py").write_text(GATE)
gate_env = dict(ENV)
gate_env["CUDA_VISIBLE_DEVICES"] = "-1"
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python",
     "/content/gate_signals.py", *[str(BASE / "runs" / n)
                                   for n in ("dev-002", "dev-003", "holdout-004")]],
    env=gate_env, capture_output=True, text=True, timeout=1800,
)
if done.returncode:
    print(done.stdout[-800:])
    print(done.stderr[-2000:])
else:
    signals = {row["case"]: row for row in json.loads(Path("/content/gate_signals.json").read_text())}
    quality = {r["case"]: r for r in recheck}
    print(f"  {'case':22s} {'stem':8s} {'stem/mix dB':>11s} {'crest dB':>9s} {'N1':>6s}")
    print("  " + "-" * 62)
    pairs = []
    for case, signal in sorted(signals.items(), key=lambda kv: kv[1]["stem_to_mix_db"]):
        row = quality.get(case)
        value = row[NEW_KEY if signal["stem"] == "guitar" else BASE_KEY]["demucs"] if row else None
        if value is None or value != value:
            continue
        pairs.append((signal["stem_to_mix_db"], value, case))
        print(f"  {case:22s} {signal['stem']:8s} {signal['stem_to_mix_db']:11.1f} "
              f"{signal['stem_crest_db']:9.1f} {value:6.3f}")
    import numpy as np
    x = np.array([p[0] for p in pairs])
    y = np.array([p[1] for p in pairs])
    print(f"\n  correlation between the stem's level and N1: {np.corrcoef(x, y)[0, 1]:+.3f}")


  case                   stem     stem/mix dB  crest dB     N1
  --------------------------------------------------------------
  mag_piano_sparse       piano          -41.3       6.2  0.076
  mag_piano_full         piano          -40.5       5.1  0.015
  karma_piano            piano          -17.1      19.4  0.460
  light_guitar_solo      guitar          -7.6      16.0  0.449
  light_guitar_rhythm    guitar          -7.6      16.0  0.605
  nice_guy_solo          guitar          -7.5      13.3  0.818
  seas_guitar_acoustic   guitar          -6.9      13.9  0.571
  seas_guitar_electric   guitar          -6.9      13.9  0.389
  karma_acoustic         guitar          -6.6      14.7  0.752
  ex_guitar_lead         guitar          -6.3      13.5  0.877
  ex_guitar_rhythm       guitar          -6.3      13.5  0.000
  nice_guy_vox           vocals          -5.6      14.7  0.659
  summer_acoustic        guitar          -5.3      16.2  0.902
  ex_bass                bass            -4.2      11

In [ ]:
"""D6b: test the gate on known-absent requests, which the finished runs already provide for free.

Every case produced all six Demucs stems, and each mixture's composition is in its config, so the
stems for instruments that were never mixed in are genuine absent-target examples.
"""

ABSENT = r'''
"""Level and crest for every Demucs stem of every case, labelled present or absent."""

import json
import re
import sys
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

RATE = 48000
STEMS = ("vocals", "drums", "bass", "guitar", "piano", "other")
# Which stem each AlbumDB source file belongs to, by its name.
SOURCE_PATTERNS = (
    ("vocals", re.compile(r"^vox")),
    ("drums", re.compile(r"^(drums|perc)")),
    ("bass", re.compile(r"^bass")),
    ("guitar", re.compile(r"^(guitar|ukelele)")),
    ("piano", re.compile(r"^keys_piano")),
    ("other", re.compile(r"^(keys|cello|viola|violin)")),
)


def category(filename):
    stem = Path(filename).stem
    for name, pattern in SOURCE_PATTERNS:
        if pattern.match(stem):
            return name
    return None


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def main():
    rows = []
    for run_dir in sys.argv[1:]:
        run = Path(run_dir)
        for case in json.loads((run / "config.json").read_text())["cases"]:
            case_dir = run / case["id"]
            if not (case_dir / "mix.wav").is_file():
                continue
            present = {category(path) for path in case["stems"]} - {None}
            mixture = mono(case_dir / "mix.wav")
            mix_rms = float(np.sqrt(np.mean(mixture**2)))
            for stem in STEMS:
                path = case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{stem}.wav"
                if not path.is_file():
                    continue
                served = mono(path, len(mixture))
                rms = float(np.sqrt(np.mean(served**2)))
                peak = float(np.max(np.abs(served)))
                rows.append({
                    "case": case["id"], "stem": stem,
                    "truth": "present" if stem in present else "absent",
                    "stem_to_mix_db": 20.0 * np.log10(max(rms, 1e-12) / max(mix_rms, 1e-12)),
                    "crest_db": 20.0 * np.log10(max(peak, 1e-12) / max(rms, 1e-12)),
                })
    Path("/content/absent_signals.json").write_text(json.dumps(rows, indent=2))
    print("ABSENT_DONE " + json.dumps({"rows": len(rows)}))


if __name__ == "__main__":
    main()
'''

Path("/content/absent_signals.py").write_text(ABSENT)
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python",
     "/content/absent_signals.py", *[str(BASE / "runs" / n)
                                     for n in ("dev-002", "dev-003", "holdout-004")]],
    env=gate_env, capture_output=True, text=True, timeout=1800,
)
if done.returncode:
    print(done.stdout[-800:])
    print(done.stderr[-2000:])
else:
    rows = json.loads(Path("/content/absent_signals.json").read_text())
    present = [r for r in rows if r["truth"] == "present"]
    absent = [r for r in rows if r["truth"] == "absent"]
    print(f"{len(present)} present, {len(absent)} absent stems across {len(rows)//6} cases\n")
    for name, group in (("present", present), ("absent", absent)):
        levels = sorted(r["stem_to_mix_db"] for r in group)
        crests = sorted(r["crest_db"] for r in group)
        print(f"  {name:8s} level  min {levels[0]:7.1f}  median {levels[len(levels)//2]:7.1f}"
              f"  max {levels[-1]:7.1f}")
        print(f"  {name:8s} crest  min {crests[0]:7.1f}  median {crests[len(crests)//2]:7.1f}"
              f"  max {crests[-1]:7.1f}")
    print()
    for threshold in (-20.0, -25.0, -30.0, -35.0):
        flagged_absent = sum(1 for r in absent if r["stem_to_mix_db"] < threshold)
        flagged_present = sum(1 for r in present if r["stem_to_mix_db"] < threshold)
        print(f"  level below {threshold:6.1f} dB -> catches {flagged_absent:3d}/{len(absent)} "
              f"absent, wrongly flags {flagged_present:2d}/{len(present)} present")


89 present, 31 absent stems across 20 cases

  present  level  min   -45.8  median    -6.8  max    -3.3
  present  crest  min     5.1  median    16.2  max    24.8
  absent   level  min   -46.4  median   -41.4  max   -37.0
  absent   crest  min     2.1  median     5.5  max    30.2

  level below  -20.0 dB -> catches  31/31 absent, wrongly flags 14/89 present
  level below  -25.0 dB -> catches  31/31 absent, wrongly flags  8/89 present
  level below  -30.0 dB -> catches  31/31 absent, wrongly flags  8/89 present
  level below  -35.0 dB -> catches  31/31 absent, wrongly flags  8/89 present


In [ ]:
"""D6c: are the eight flagged present stems false alarms, or cases we genuinely cannot deliver?"""

THRESHOLD_DB = -25.0
rows = json.loads(Path("/content/absent_signals.json").read_text())
quality = {r["case"]: r for r in recheck}
requested = {}
for name in ("dev-002", "dev-003", "holdout-004"):
    for case in json.loads((BASE / "runs" / name / "config.json").read_text())["cases"]:
        requested[case["id"]] = case["prompt"]

flagged = [r for r in rows if r["truth"] == "present" and r["stem_to_mix_db"] < THRESHOLD_DB]
print(f"{len(flagged)} present stems fall below {THRESHOLD_DB:.0f} dB\n")
print(f"  {'case':22s} {'stem':8s} {'level':>7s} {'crest':>7s} {'was it requested?':>18s} {'N1':>7s}")
print("  " + "-" * 74)
for row in sorted(flagged, key=lambda r: r["stem_to_mix_db"]):
    prompt = requested.get(row["case"], "")
    asked = row["stem"] in prompt.lower() or (
        row["stem"] == "piano" and "piano" in prompt.lower()
    )
    measured = quality.get(row["case"])
    value = ""
    if asked and measured:
        key = NEW_KEY if row["stem"] == "guitar" else BASE_KEY
        score = measured[key].get("demucs")
        value = f"{score:.3f}" if score == score else ""
    print(f"  {row['case']:22s} {row['stem']:8s} {row['stem_to_mix_db']:7.1f} "
          f"{row['crest_db']:7.1f} {('yes, ' + prompt) if asked else 'no':>18s} {value:>7s}")

present = [r for r in rows if r["truth"] == "present"]
absent = [r for r in rows if r["truth"] == "absent"]
deliverable = [r for r in present if r["stem_to_mix_db"] >= THRESHOLD_DB]
print(f"\n  gate at {THRESHOLD_DB:.0f} dB:")
print(f"    absent stems caught            {sum(1 for r in absent if r['stem_to_mix_db'] < THRESHOLD_DB)}/{len(absent)}")
print(f"    deliverable stems passed       {len(deliverable)}/{len(present)}")
print(f"    present but unrecoverable      {len(flagged)} (shown above)")


8 present stems fall below -25 dB

  case                   stem       level   crest  was it requested?      N1
  --------------------------------------------------------------------------
  ex_guitar_lead         vocals     -45.8    15.3                 no        
  ex_guitar_rhythm       vocals     -45.8    15.3                 no        
  ex_bass                vocals     -45.8    15.3                 no        
  mag_piano_sparse       piano      -41.3     6.2         yes, piano   0.076
  mag_piano_full         piano      -40.5     5.1         yes, piano   0.015
  diva_synth             other      -39.9    21.8                 no        
  summer_acoustic        other      -35.0    24.5                 no        
  summer_synth           other      -35.0    24.5                 no        

  gate at -25 dB:
    absent stems caught            31/31
    deliverable stems passed       81/89
    present but unrecoverable      8 (shown above)


## Where a lost piano goes


In [ ]:
"""P1: when Demucs' piano stem comes back empty, which stem swallowed the piano?

If a different stem carries it, a routing or summing rule can recover it. If no stem does, the
piano is lost inside the model and only a better separation pass can help.
"""

WHERE = r'''
import json
import sys
from itertools import combinations
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf

RATE = 48000
STEMS = ("vocals", "drums", "bass", "guitar", "piano", "other")


def mono(path, length=None):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    samples = audio.mean(axis=1)
    if rate != RATE:
        samples = librosa.resample(samples, orig_sr=rate, target_sr=RATE)
    samples = np.asarray(samples, dtype=np.float64)
    return samples if length is None else samples[:length]


def si_sdr(reference, estimate):
    n = min(len(reference), len(estimate))
    target, audio = reference[:n] - reference[:n].mean(), estimate[:n] - estimate[:n].mean()
    power = float(target @ target)
    if power < 1e-12:
        return float("nan")
    projection = target * float(audio @ target) / power
    error = audio - projection
    return 10 * np.log10((projection @ projection + 1e-12) / (error @ error + 1e-12))


for case_dir in (Path(p) for p in sys.argv[1:]):
    reference = mono(case_dir / "reference.wav")
    mixture = mono(case_dir / "mix.wav", len(reference))
    stems = {s: mono(case_dir / "demucs" / "htdemucs_6s" / "mix" / f"{s}.wav", len(reference))
             for s in STEMS}
    print(f"{case_dir.name}: piano vs mixture {si_sdr(reference, mixture):+.1f} dB")
    single = {s: si_sdr(reference, w) for s, w in stems.items()}
    for stem, value in sorted(single.items(), key=lambda kv: -kv[1]):
        print(f"    {stem:8s} {value:+7.1f} dB")
    best_pair = max(combinations(STEMS, 2),
                    key=lambda pair: si_sdr(reference, stems[pair[0]] + stems[pair[1]]))
    print(f"    best pair {'+'.join(best_pair):16s} "
          f"{si_sdr(reference, stems[best_pair[0]] + stems[best_pair[1]]):+7.1f} dB")
    # What fraction of the piano's spectrum is left in the residual the mix minus all stems?
    leftover = mixture - sum(stems.values())
    print(f"    mixture minus all six stems {si_sdr(reference, leftover):+7.1f} dB\n")
'''

Path("/content/where.py").write_text(WHERE)
cases = [BASE / "runs" / "dev-002" / "mag_piano_sparse",
         BASE / "runs" / "dev-002" / "mag_piano_full",
         BASE / "runs" / "holdout-004" / "karma_piano"]
done = subprocess.run(
    [UV, "run", "--no-sync", "--project", str(PROJECTS["pitch"]), "python", "/content/where.py",
     *map(str, cases)],
    env=gate_env, capture_output=True, text=True, timeout=1800,
)
print(done.stdout[-3000:] if done.returncode == 0 else done.stderr[-2500:])


mag_piano_sparse: piano vs mixture -24.2 dB
    guitar     -16.3 dB
    piano      -17.8 dB
    other      -18.7 dB
    vocals     -29.2 dB
    bass       -44.4 dB
    drums      -53.9 dB
    best pair guitar+piano       -16.3 dB
    mixture minus all six stems   -17.8 dB

mag_piano_full: piano vs mixture -24.9 dB
    guitar     -23.3 dB
    vocals     -23.9 dB
    other      -25.3 dB
    piano      -31.4 dB
    bass       -50.4 dB
    drums      -50.7 dB
    best pair vocals+guitar      -21.7 dB
    mixture minus all six stems   -32.3 dB

karma_piano: piano vs mixture -14.0 dB
    piano       -4.5 dB
    guitar     -17.2 dB
    bass       -22.4 dB
    other      -34.6 dB
    drums      -40.4 dB
    vocals     -47.1 dB
    best pair piano+other         -4.4 dB
    mixture minus all six stems   -20.6 dB
